# Reduced Data Pipeline

### Thie notebook is a reduced version of the main Data Pipeline Notebook. More precisely, this notebook does not compute any of the high cardinality data | we are utilizng the full dataset

March 5th, 2025

Maxime Bouthillier

### Importing Libraries and Specialty Functions

In [2]:
import pandas as pd
import os 
import glob 
from datetime import datetime
import warnings
from joblib import dump

# Importing Specialized Functions
import ipynb.fs.full.Pipeline_Functions as func

# Data Cleaning

### Admissions

In [3]:
# Setting the Directory
directory = '/Users/maxb/Library/CloudStorage/OneDrive-UniversityofWaterloo/Hospital Research/Datasets/MIMI-III_Full'
os.chdir(directory)


# Reading the csv file
adm_df = pd.read_csv("ADMISSIONS.csv")
adm_df.columns = adm_df.columns.str.lower()


# Cleaning the time based features
adm_df["edregtime"] = adm_df["edregtime"].fillna("1677-09-22 00:00:00")                                             
adm_df["edouttime"] = adm_df["edouttime"].fillna("1677-09-22 00:00:00")

adm_df = func.as_datetime(adm_df, column='admittime')
adm_df = func.as_datetime(adm_df, column='dischtime')
adm_df = func.as_datetime(adm_df, column='edregtime')
adm_df = func.as_datetime(adm_df, column='edouttime')  


# Removing all admission instances where a patient died
adm_df = adm_df.drop(adm_df[adm_df['hospital_expire_flag'] == 1].index)


# Setting Marital Status to binary variables
adm_df['marital_status'] = adm_df['marital_status'].apply(lambda x: 1 if x == "MARRIED" else 0)

In [4]:
# Creating Readmission Feature and subsetting the dataset
adm_df = func.readmission(adm_df, 30)

# Admission Duration time feature
adm_df['admit_duration'] = adm_df['dischtime'] - adm_df['admittime']

# ED Duration time feature
adm_df['ed_duration'] = adm_df['edouttime'] - adm_df['edregtime']


# Removing unnecessary variables
col_drops = ["row_id", "language", "religion", "hospital_expire_flag", "hadm_id", 
             "has_chartevents_data", "edregtime", "edouttime", "deathtime", "diagnosis"]

for i in col_drops:
    adm_df = adm_df.drop(i, axis=1)

# Checing NaN instances
func.check_nan(adm_df)

### Patients

In [5]:
# Reading the csv file
pat_df = pd.read_csv("PATIENTS.csv")
pat_df.columns = pat_df.columns.str.lower()


# Selecting only the necessary columns
pat_df = pat_df[['subject_id', 'gender', 'dob']]


# Double checking that there are no NaN values
func.check_nan(pat_df)

### ICU LOS

In [6]:
# Reading the csv file
icu_df = pd.read_csv("ICUSTAYS.csv")
icu_df.columns = icu_df.columns.str.lower()
icu_df = icu_df.dropna()


# Cleaning the time based features
icu_df  = func.as_datetime(icu_df , column='intime')
icu_df = func.as_datetime(icu_df , column='outtime')


# Subsetting the dataframe by only the releveat subject_ID entries:
icu_df = func.subject_subset(icu_df , adm_df, column='intime')

In [7]:
# Keeping only the necessary columns
icu_df  = icu_df[['subject_id', 'los']]

# Combining discontinuous ICU stays
icu_df = icu_df.groupby('subject_id', as_index=False).agg({'los': 'sum'})

# Double checking that there are no NaN values
func.check_nan(icu_df)

### Merging Dataframes

In [12]:
# Left join of Patients table on Admission table
master_df = pd.merge(adm_df, pat_df, how='inner', on='subject_id')

# Left join of ICU LOS table on df
master_df = pd.merge(master_df, icu_df, how='inner', on='subject_id')

master_df = master_df.fillna(0)

dump(master_df, "Master_Dataframe_reduced.joblib")

['Master_Dataframe_reduced.joblib']

In [11]:
master_df.head()

,subject_id,admittime,dischtime,admission_type,admission_location,discharge_location,insurance,marital_status,ethnicity,read_flag,admit_duration,ed_duration,gender,dob,los
0,82574,2100-06-07 19:59:00,2100-06-09 17:09:00,EMERGENCY,CLINIC REFERRAL/PREMATURE,HOME,Medicaid,0,OTHER,0.0,1 days 21:10:00,0 days 10:52:00,M,2044-04-23 00:00:00,0.7911
1,12001,2100-06-14 04:55:00,2100-06-27 12:00:00,EMERGENCY,EMERGENCY ROOM ADMIT,SNF,Medicare,1,WHITE,0.0,13 days 07:05:00,0 days 02:07:00,F,2028-10-27 00:00:00,4.6201
2,21081,2100-06-14 12:02:00,2100-06-17 14:20:00,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Medicaid,0,BLACK/AFRICAN AMERICAN,0.0,3 days 02:18:00,0 days 06:49:00,F,2067-04-01 00:00:00,1.1269
3,32096,2100-06-22 03:04:00,2100-06-30 11:35:00,EMERGENCY,EMERGENCY ROOM ADMIT,REHAB/DISTINCT PART HOSP,Private,1,WHITE,0.0,8 days 08:31:00,0 days 08:34:00,F,2070-12-13 00:00:00,2.2924
4,20957,2100-06-24 22:37:00,2100-07-03 12:31:00,EMERGENCY,EMERGENCY ROOM ADMIT,HOME HEALTH CARE,Private,1,WHITE,0.0,8 days 13:54:00,0 days 10:33:00,F,2052-04-04 00:00:00,1.9028
